# BirdCLEF 2026 — Exploratory Data Analysis
Goals:
1. Class distribution (how many recordings per species?)
2. Audio duration distribution
3. Label quality (secondary labels, rating distribution)
4. Soundscape label coverage
5. Sample spectrogram visualizations

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

import torchaudio
import torch

from config import Config, ALL_CLASSES, CLASS_TO_IDX, NUM_CLASSES
from audio_utils import load_audio, build_mel_transform, waveform_to_logmel, crop_or_pad

cfg = Config()
print(f'NUM_CLASSES: {NUM_CLASSES}')

## 1. Training metadata

In [ ]:
train_df = pd.read_csv(cfg.train_csv)
tax_df   = pd.read_csv(cfg.taxonomy_csv)
sc_df    = pd.read_csv(cfg.soundscape_labels_csv)

print('train.csv shape:', train_df.shape)
print('taxonomy.csv shape:', tax_df.shape)
print('soundscape_labels.csv shape:', sc_df.shape)
train_df.head(3)

## 2. Class distribution

In [ ]:
counts = train_df['primary_label'].value_counts()
print(f'Classes with data: {len(counts)} / {NUM_CLASSES}')
print(f'Min samples: {counts.min()} ({counts.idxmin()})')
print(f'Max samples: {counts.max()} ({counts.idxmax()})')
print(f'Median samples: {counts.median():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
counts.plot(kind='bar', ax=axes[0], width=0.8, color='steelblue')
axes[0].set_title('Recordings per species (sorted)')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')
axes[0].set_xticks([])

counts.plot(kind='hist', bins=40, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of recordings per species')
axes[1].set_xlabel('Number of recordings')
plt.tight_layout()
plt.show()

## 3. Class breakdown by taxonomic class

In [ ]:
merged = train_df.merge(tax_df[['primary_label','class_name']], on='primary_label', how='left')
class_counts = merged.groupby('class_name').size().sort_values(ascending=False)
print(class_counts)

class_counts.plot(kind='bar', color='coral', figsize=(8, 4), edgecolor='white')
plt.title('Recordings by taxonomic class')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Audio duration distribution (sample 500 files)

In [ ]:
sample_rows = train_df.sample(min(500, len(train_df)), random_state=42)
durations = []
for _, row in sample_rows.iterrows():
    path = cfg.train_audio_dir / row['filename']
    if path.exists():
        info = torchaudio.info(str(path))
        dur  = info.num_frames / info.sample_rate
        durations.append(dur)

durations = np.array(durations)
print(f'Duration stats (seconds): min={durations.min():.1f}  median={np.median(durations):.1f}  max={durations.max():.1f}')

plt.figure(figsize=(8, 4))
plt.hist(durations, bins=40, color='steelblue', edgecolor='white')
plt.axvline(5, color='red', linestyle='--', label='5s training window')
plt.xlabel('Duration (s)')
plt.ylabel('Count')
plt.title('Audio duration distribution (500-sample)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Rating distribution

In [ ]:
plt.figure(figsize=(6, 3))
train_df['rating'].hist(bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.title('Recording quality rating distribution')
plt.tight_layout()
plt.show()
print(train_df['rating'].describe())

## 6. Soundscape label co-occurrence

In [ ]:
sc_df['n_species'] = sc_df['primary_label'].apply(lambda x: len(str(x).split(';')))
print('Species per 5s window:')
print(sc_df['n_species'].describe())

plt.figure(figsize=(6, 3))
sc_df['n_species'].hist(bins=20, color='coral', edgecolor='white')
plt.xlabel('Number of co-occurring species')
plt.ylabel('Count')
plt.title('Species per soundscape 5s window')
plt.tight_layout()
plt.show()

## 7. Sample log-mel spectrogram visualizations

In [ ]:
mel_transform = build_mel_transform(cfg)

sample_rows = train_df.groupby('primary_label').first().sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(15, 6))

for ax, (label, row) in zip(axes.flat, sample_rows.iterrows()):
    path = cfg.train_audio_dir / row['filename']
    if not path.exists():
        ax.set_title(f'{label}\n(file missing)')
        continue
    waveform = load_audio(path, cfg)
    waveform = crop_or_pad(waveform, cfg.clip_samples)
    log_mel  = waveform_to_logmel(waveform, mel_transform).squeeze(0).numpy()
    im = ax.imshow(log_mel, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'{label}\n({row["common_name"] if "common_name" in row else ""})', fontsize=8)
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Mel bin')

plt.suptitle('Log-mel spectrograms — one sample per species', y=1.01)
plt.tight_layout()
plt.show()